<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/3_dise%C3%B1o_entrenamiento_evaluacion/3_2_Generacion_X_y.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2. Generación de X e y para modelos

## 0. Clonado de Repositorio, instalación de librería e importación.

### Clonado de Repositorio

In [30]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


### Acceso de Drive

In [31]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Instalación de librerías

In [32]:
# !{sys.executable} -m pip install -q pandas_market_calendars  # Solo si usás horarios de mercados
!{sys.executable} -m pip install -q ta
print("Librerías instaladas: ta")

Librerías instaladas: ta


### Importación de librerías

In [33]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Utilidades del sistema y fechas
import os
import glob
import requests
from datetime import datetime, timedelta

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Análisis técnico
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

# Estadística
from scipy.stats import spearmanr

#que es?
#from tqdm.notebook import tqdm
from tqdm import tqdm
# Modelos ML
#from xgboost import XGBRegressor
#from sklearn.metrics import mean_squared_error, r2_score

# Calendario de mercados (descomentar si lo necesitás)
# import pandas_market_calendars as mcal

## 1. Carga de datasets train, valid y test.

In [34]:
def load_data_from_drive():
    """
    Función para cargar un archivo Parquet desde el drive
        """
    # Definir la URL del archivo Parquet en Drive
    df_path_mnq = f'{drive_path}/mnq_data/mnq_model.parquet'
    df_path_train = f'{drive_path}/mnq_data/mnq_train.parquet'
    df_path_valid = f'{drive_path}/mnq_data/mnq_valid.parquet'
    df_path_test = f'{drive_path}/mnq_data/mnq_test.parquet'
    df_path_factores = f'{drive_path}/df_factores.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [35]:
def load_data_from_repo():
    """
    Función para cargar un archivo Parquet desde el repositorio clonado
    """
    # Definir la URL del archivo Parquet en GitHub
    df_path_mnq = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_model.parquet'
    df_path_factores = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/df_factores.parquet'
    df_path_train = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_train.parquet'
    df_path_valid = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_valid.parquet'
    df_path_test = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_test.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [36]:
mnq_model, indicadores_tecnicos, mnq_train, mnq_valid, mnq_test = load_data_from_drive()

## 1.1. Información de los datasets

In [37]:
def info_dataset (df): # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"Cantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['target_return_30']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"Valores por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo
  print(f"Hora diaria de inicio {primer_hora}")
  print(f"Hora diaria de final {ultima_hora}")
  print(f"Zona horaria: {zona_horaria}")

In [38]:
info_dataset(mnq_train)

Cantidad de días: 917
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [39]:
info_dataset(mnq_valid)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [40]:
info_dataset(mnq_test)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


## 2. Generación de ventanas X e y

In [41]:
target_column = "target_return_30"
features = mnq_model.columns.tolist()
features.remove(target_column)
features.remove('date')

window_size = 60

In [42]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue
            vector = ventana.values.flatten()
            target = grupo.loc[i+window_size-1, target_col]
            X.append(vector)
            y.append(target)
    return np.array(X), np.array(y)

### Generamos los X_* e y_* con el total de features

In [43]:
features

['open',
 'high',
 'low',
 'close',
 'volume',
 'momentum_3',
 'momentum_10',
 'roc_5',
 'roc_20',
 'rsi_3',
 'rsi_7',
 'rsi_14',
 'stoch_k_20',
 'bb_percent_20_15',
 'bb_percent_30_20',
 'price_ema30',
 'atr_norm',
 'reversal_momentum_factor',
 'reversal_media_factor',
 'reversion_vol_momentum_factor',
 'factor30']

Antes de correr la generación de X e y, tenemos que verificar si es que no existe en la carpeta ventanas_X_y:


In [44]:
# Subcarpeta donde querés guardar
save_dir = f"{drive_path}/ventanas_x_y"

# Crear carpeta si no existe
os.makedirs(save_dir, exist_ok=True)


In [45]:
#Ruta de x_y
ruta_x_y_train = f"{drive_path}/ventanas_x_y/mnq_Xy_train.npz"
ruta_x_y_valid = f"{drive_path}/ventanas_x_y/mnq_Xy_valid.npz"
ruta_x_y_test = f"{drive_path}/ventanas_x_y/mnq_Xy_test.npz"


#### Para X_train e y_train

In [48]:
if not os.path.exists(ruta_x_y_train):
    print('El archivo no existe -> Generando X_train e y_train: ')
    X_train, y_train = generar_ventanas(mnq_train, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_train, X=X_train, y=y_train)
    print("Guardado:", ruta_x_y_train)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_train)
    data_train = np.load(ruta_x_y_train)
    X_train, y_train = data_train["X"], data_train["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_train.npz


In [49]:
print(f"X_train info: {X_train.shape[0]} ventanas aplanadas en {X_train.shape[1]} números, el equivalente a la window size ({window_size}) por la cantidad de features ({len(features)})")
print(f"y_train info: {y_train.shape[0]} valores que corresponden al retorno a 30 minutos ({target_column})")

X_train info: 276017 ventanas aplanadas en 1260 números, el equivalente a la window size (60) por la cantidad de features (21)
y_train info: 276017 valores que corresponden al retorno a 30 minutos (target_return_30)


#### Para X_valid e y_valid

In [50]:
if not os.path.exists(ruta_x_y_valid):
    print('El archivo no existe -> Generando X_valid e y_valid: ')
    X_valid, y_valid = generar_ventanas(mnq_valid, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_valid, X=X_valid, y=y_valid)
    print("Guardado:", ruta_x_y_valid)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_valid)
    data_valid = np.load(ruta_x_y_valid)
    X_valid, y_valid = data_valid["X"], data_valid["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_valid.npz


#### Para X_test e y_test

In [51]:
if not os.path.exists(ruta_x_y_test):
    print('El archivo no existe -> Generando X_test e y_test: ')
    X_test, y_test = generar_ventanas(mnq_test, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_test, X=X_test, y=y_test)
    print("Guardado:", ruta_x_y_test)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_test)
    data_test = np.load(ruta_x_y_test)
    X_test, y_test = data_test["X"], data_test["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_test.npz
